
# 02 – Structured Outputs: GCP Architecture Review

## Objective

Extract structured GCP architecture requirements from natural language using LLMs, JSON and Pydantic.

## Workflow

1. Cloud Architecture Requirement
2. LLM Extraction (Groq)
3. JSON Output
4. Pydantic Validation
5. Error Handling and Retry
6. Batch Evaluation
7. Human Review Decision

## Learning Goals

- Define structured data contracts
- Compare plain prompting and JSON Mode
- Validate LLM outputs with Pydantic
- Detect syntax and schema errors
- Implement retry logic
- Evaluate extracted requirements
- Apply deterministic human review policies


In [31]:
import json
from enum import Enum
from typing import Literal

from pydantic import BaseModel, Field, ValidationError


In [3]:
class CloudService(str, Enum):
    GKE = "GKE"
    CLOUD_RUN = "CLOUD_RUN"
    CLOUD_SQL = "CLOUD_SQL"
    VERTEX_AI = "VERTEX_AI"
    CLOUD_STORAGE = "CLOUD_STORAGE"


class CloudArchitecture(BaseModel):
    environment: Literal["dev", "test", "prod"]
    region: str
    services: list[CloudService]
    high_availability: bool
    monitoring_required: bool
    confidence: float = Field(ge=0, le=1)

In [4]:
architecture = CloudArchitecture(
    environment="prod",
    region="europe-west3",
    services=["GKE", "CLOUD_SQL"],
    high_availability=True,
    monitoring_required=True,
    confidence=0.95
)

print(architecture.model_dump_json(indent=2))

{
  "environment": "prod",
  "region": "europe-west3",
  "services": [
    "GKE",
    "CLOUD_SQL"
  ],
  "high_availability": true,
  "monitoring_required": true,
  "confidence": 0.95
}


## pydantic creates a JSON schema contract from the python class

In [5]:
import json

schema = CloudArchitecture.model_json_schema()

print(json.dumps(schema, indent=2))

{
  "$defs": {
    "CloudService": {
      "enum": [
        "GKE",
        "CLOUD_RUN",
        "CLOUD_SQL",
        "VERTEX_AI",
        "CLOUD_STORAGE"
      ],
      "title": "CloudService",
      "type": "string"
    }
  },
  "properties": {
    "environment": {
      "enum": [
        "dev",
        "test",
        "prod"
      ],
      "title": "Environment",
      "type": "string"
    },
    "region": {
      "title": "Region",
      "type": "string"
    },
    "services": {
      "items": {
        "$ref": "#/$defs/CloudService"
      },
      "title": "Services",
      "type": "array"
    },
    "high_availability": {
      "title": "High Availability",
      "type": "boolean"
    },
    "monitoring_required": {
      "title": "Monitoring Required",
      "type": "boolean"
    },
    "confidence": {
      "maximum": 1,
      "minimum": 0,
      "title": "Confidence",
      "type": "number"
    }
  },
  "required": [
    "environment",
    "region",
    "services",
    "high_a

In [6]:
from pydantic import ValidationError

invalid_architecture = {
    "environment": "production",
    "region": "europe-west3",
    "services": ["GKE", "UNKNOWN_SERVICE"],
    "high_availability": True,
    "monitoring_required": True,
    "confidence": 1.5
}

try:
    architecture = CloudArchitecture.model_validate(
        invalid_architecture
    )
    print("Validation successful:", architecture)

except ValidationError as error:
    print("Validation failed!\n")

    for issue in error.errors():
        print("Field:", issue["loc"])
        print("Error:", issue["msg"])
        print("---")

Validation failed!

Field: ('environment',)
Error: Input should be 'dev', 'test' or 'prod'
---
Field: ('services', 1)
Error: Input should be 'GKE', 'CLOUD_RUN', 'CLOUD_SQL', 'VERTEX_AI' or 'CLOUD_STORAGE'
---
Field: ('confidence',)
Error: Input should be less than or equal to 1
---


In [7]:
from prompt_helpers import client_and_model

client, model, provider, dry_reason = client_and_model()

print("Provider:", provider.name)
print("Model:", model)
print("Mode:", "LIVE" if client else "DRY RUN")

Provider: groq
Model: openai/gpt-oss-20b
Mode: LIVE


## Cloud Architecture Prompt

In [8]:
cloud_requirement = """
Deploy a production ML inference API on Google Cloud
in europe-west3.

Use GKE for container orchestration and Cloud SQL
for the PostgreSQL database.

The architecture must support high availability
and include monitoring.
"""

system_prompt = """
You are a Google Cloud Solution Architect.

Extract architecture requirements from the user text.

Return ONLY a valid JSON object with these fields:

- environment: dev, test, or prod
- region: GCP region
- services: list of GCP services
- high_availability: boolean
- monitoring_required: boolean
- confidence: number between 0 and 1

Allowed services:
GKE, CLOUD_RUN, CLOUD_SQL, VERTEX_AI, CLOUD_STORAGE

Do not invent services or requirements.
"""

## First call

In [9]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": cloud_requirement},
    ],
    temperature=0,
    response_format={"type": "json_object"},
)

raw_output = response.choices[0].message.content

print(raw_output)

{"environment":"prod","region":"europe-west3","services":["GKE","CLOUD_SQL"],"high_availability":true,"monitoring_required":true,"confidence":0.95}


##. LLM Output  validation with pydantic

In [10]:
from pydantic import ValidationError

try:
    validated_architecture = CloudArchitecture.model_validate_json(
        raw_output
    )

    print("✅ Architecture validation successful!\n")
    print(validated_architecture.model_dump_json(indent=2))

except ValidationError as error:
    print("❌ Architecture validation failed!")
    print(error)

✅ Architecture validation successful!

{
  "environment": "prod",
  "region": "europe-west3",
  "services": [
    "GKE",
    "CLOUD_SQL"
  ],
  "high_availability": true,
  "monitoring_required": true,
  "confidence": 0.95
}


## Validation Helper

In [11]:
def validate_architecture(raw_json: str):
    try:
        architecture = CloudArchitecture.model_validate_json(
            raw_json
        )
        return architecture, None

    except ValidationError as error:
        return None, error

In [12]:
invalid_response = """
{
    "environment": "production",
    "region": "europe-west3",
    "services": ["GKE", "KUBERNETES"],
    "high_availability": true,
    "monitoring_required": true,
    "confidence": 1.2
}
"""

architecture, error = validate_architecture(invalid_response)

if error:
    print("❌ Validation failed!\n")
    for issue in error.errors():
        print(f"Field: {issue['loc']}")
        print(f"Reason: {issue['msg']}")
        print("---")
else:
    print("✅ Validation successful!")

❌ Validation failed!

Field: ('environment',)
Reason: Input should be 'dev', 'test' or 'prod'
---
Field: ('services', 1)
Reason: Input should be 'GKE', 'CLOUD_RUN', 'CLOUD_SQL', 'VERTEX_AI' or 'CLOUD_STORAGE'
---
Field: ('confidence',)
Reason: Input should be less than or equal to 1
---


## LLM Retry mit Groq

In [13]:
def repair_architecture(raw_json: str, requirement: str, max_retries=2):

    for attempt in range(max_retries + 1):

        architecture, error = validate_architecture(raw_json)

        if error is None:
            print(f"✅ Validation successful (attempt {attempt + 1})")
            return architecture

        print(f"❌ Validation failed (attempt {attempt + 1})")

        if attempt == max_retries:
            print("Maximum retries reached.")
            return None

        correction_prompt = f"""
        You are a Google Cloud Solution Architect.

        Original requirement:
        {requirement}

        Invalid JSON:
        {raw_json}

        Validation errors:
        {error}

        Correct the JSON to match this schema:
        {json.dumps(CloudArchitecture.model_json_schema())}

        Preserve the original requirements.
        Do not invent services or requirements.
        Return only valid JSON.
        """

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": correction_prompt}
            ],
            temperature=0,
            response_format={"type": "json_object"},
        )

        raw_json = response.choices[0].message.content

    return None

In [14]:
repaired_architecture = repair_architecture(
    raw_json=invalid_response,
    requirement=cloud_requirement,
    max_retries=2
)

if repaired_architecture:
    print("\nFinal validated architecture:")
    print(repaired_architecture.model_dump_json(indent=2))

❌ Validation failed (attempt 1)
✅ Validation successful (attempt 2)

Final validated architecture:
{
  "environment": "prod",
  "region": "europe-west3",
  "services": [
    "GKE",
    "CLOUD_SQL"
  ],
  "high_availability": true,
  "monitoring_required": true,
  "confidence": 1.0
}


### Batch Extraction

In [15]:
cloud_requirements = [
    """
    Deploy a development API using Cloud Run in europe-west3.
    Monitoring is required, but high availability is not mandatory.
    """,
    """
    Deploy a test environment using GKE and Cloud Storage
    in europe-west3. Monitoring is required.
    High availability is not required.
    """,
    """
    Deploy a production ML inference platform using GKE
    and Cloud SQL in europe-west3.
    High availability and monitoring are mandatory.
    """
]

print(f"Architecture requirements: {len(cloud_requirements)}")

Architecture requirements: 3


In [16]:
def extract_architectures(requirements):
    results = []

    for index, requirement in enumerate(requirements, start=1):
        print(f"\nProcessing architecture {index}...")

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": requirement},
            ],
            temperature=0,
            response_format={"type": "json_object"},
        )

        raw_json = response.choices[0].message.content

        architecture = repair_architecture(
            raw_json=raw_json,
            requirement=requirement,
            max_retries=2,
        )

        results.append(architecture)

    return results

In [17]:
architectures = extract_architectures(cloud_requirements)

for index, architecture in enumerate(architectures, start=1):
    print(f"\nArchitecture {index}")

    if architecture is not None:
        print(architecture.model_dump_json(indent=2))
    else:
        print("Validation failed")


Processing architecture 1...
✅ Validation successful (attempt 1)

Processing architecture 2...
✅ Validation successful (attempt 1)

Processing architecture 3...
✅ Validation successful (attempt 1)

Architecture 1
{
  "environment": "dev",
  "region": "europe-west3",
  "services": [
    "CLOUD_RUN"
  ],
  "high_availability": false,
  "monitoring_required": true,
  "confidence": 0.95
}

Architecture 2
{
  "environment": "test",
  "region": "europe-west3",
  "services": [
    "GKE",
    "CLOUD_STORAGE"
  ],
  "high_availability": false,
  "monitoring_required": true,
  "confidence": 0.95
}

Architecture 3
{
  "environment": "prod",
  "region": "europe-west3",
  "services": [
    "GKE",
    "CLOUD_SQL"
  ],
  "high_availability": true,
  "monitoring_required": true,
  "confidence": 0.95
}


## Human Review Decision

In [18]:
def requires_human_review(architecture: CloudArchitecture) -> bool:

    # Low confidence
    if architecture.confidence < 0.8:
        return True

    # Production always requires review
    if architecture.environment == "prod":
        return True

    # GKE architectures require review
    if CloudService.GKE in architecture.services:
        return True

    return False

In [19]:
for index, architecture in enumerate(architectures, start=1):

    if architecture is None:
        print(f"Architecture {index}: VALIDATION FAILED")
        continue

    review = requires_human_review(architecture)

    status = "HUMAN REVIEW" if review else "AUTOMATED PROCESSING"

    print(
        f"Architecture {index} | "
        f"Environment: {architecture.environment} | "
        f"Decision: {status}"
    )

Architecture 1 | Environment: dev | Decision: AUTOMATED PROCESSING
Architecture 2 | Environment: test | Decision: HUMAN REVIEW
Architecture 3 | Environment: prod | Decision: HUMAN REVIEW


## Plain Prompting vs. JSON Mode

In [20]:
plain_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": f"""
            Extract GCP architecture requirements.

            Return environment, region, services,
            high_availability, monitoring_required,
            and confidence.

            Requirement:
            {cloud_requirement}
            """
        }
    ],
    temperature=0
)

plain_output = plain_response.choices[0].message.content

print(plain_output)

```json
{
  "environment": "production",
  "region": "europe-west3",
  "services": [
    "Google Kubernetes Engine (GKE)",
    "Cloud SQL (PostgreSQL)",
    "Cloud Load Balancing",
    "Cloud Monitoring (Operations Suite)",
    "Cloud Logging",
    "Cloud IAM & Service Accounts",
    "Cloud Build / Artifact Registry (optional for CI/CD)"
  ],
  "high_availability": true,
  "monitoring_required": true,
  "confidence": "high"
}
```


## Testing JSON Syntax

In [21]:
malformed_json = """
{
    "environment": "prod",
    "region": "europe-west3",
    "services": ["GKE", "CLOUD_SQL"],
    "high_availability": true,
    "monitoring_required": true,
    "confidence": 0.95,
}
"""

architecture, error = validate_architecture(malformed_json)

if error:
    print("❌ Invalid JSON detected!")
    print(error)
else:
    print("✅ Validation successful!")

❌ Invalid JSON detected!
1 validation error for CloudArchitecture
  Invalid JSON: trailing comma at line 9 column 1 [type=json_invalid, input_value='\n{\n    "environment": ...confidence": 0.95,\n}\n', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid


## Batch Evaluation

In [25]:
expected_results = [
    {
        "environment": "dev",
        "services": {"CLOUD_RUN"},
        "high_availability": False,
        "monitoring_required": True,
    },
    {
        "environment": "test",
        "services": {"GKE", "CLOUD_STORAGE"},
        "high_availability": False,
        "monitoring_required": True,
    },
    {
        "environment": "prod",
        "services": {"GKE", "CLOUD_SQL"},
        "high_availability": True,
        "monitoring_required": True,
    },
]

In [26]:
def evaluate_architectures(architectures, expected_results):
    if len(architectures) != len(expected_results):
        raise ValueError("Evaluation dataset size mismatch")

    passed = 0

    for index, (actual, expected) in enumerate(
        zip(architectures, expected_results), start=1
    ):
        if actual is None:
            print(f"Architecture {index}: FAIL (validation)")
            continue

        checks = {
            "environment": actual.environment == expected["environment"],
            "services": (
                {service.value for service in actual.services}
                == expected["services"]
            ),
            "high_availability": (
                actual.high_availability == expected["high_availability"]
            ),
            "monitoring_required": (
                actual.monitoring_required == expected["monitoring_required"]
            ),
        }

        success = all(checks.values())
        passed += int(success)

        print(f"Architecture {index}: {'PASS' if success else 'FAIL'}")

        for field, result in checks.items():
            if not result:
                print(f"  Mismatch: {field}")

    print(f"\nEvaluation: {passed}/{len(expected_results)} passed")

In [27]:
evaluate_architectures(architectures, expected_results)

Architecture 1: PASS
Architecture 2: PASS
Architecture 3: PASS

Evaluation: 3/3 passed


## Human Review im Structured Output

In [28]:
class ArchitectureReviewResult(BaseModel):
    architecture: CloudArchitecture
    requires_human_review: bool
    review_reason: str

In [29]:
def create_review_result(architecture: CloudArchitecture):

    reasons = []

    if architecture.confidence < 0.8:
        reasons.append("Low confidence")

    if architecture.environment == "prod":
        reasons.append("Production environment")

    if CloudService.GKE in architecture.services:
        reasons.append("GKE requires review")

    return ArchitectureReviewResult(
        architecture=architecture,
        requires_human_review=bool(reasons),
        review_reason=", ".join(reasons) if reasons else "No review required",
    )

In [30]:
review_results = [
    create_review_result(architecture)
    for architecture in architectures
    if architecture is not None
]

for index, result in enumerate(review_results, start=1):
    print(f"\nArchitecture {index}")
    print(result.model_dump_json(indent=2))


Architecture 1
{
  "architecture": {
    "environment": "dev",
    "region": "europe-west3",
    "services": [
      "CLOUD_RUN"
    ],
    "high_availability": false,
    "monitoring_required": true,
    "confidence": 0.95
  },
  "requires_human_review": false,
  "review_reason": "No review required"
}

Architecture 2
{
  "architecture": {
    "environment": "test",
    "region": "europe-west3",
    "services": [
      "GKE",
      "CLOUD_STORAGE"
    ],
    "high_availability": false,
    "monitoring_required": true,
    "confidence": 0.95
  },
  "requires_human_review": true,
  "review_reason": "GKE requires review"
}

Architecture 3
{
  "architecture": {
    "environment": "prod",
    "region": "europe-west3",
    "services": [
      "GKE",
      "CLOUD_SQL"
    ],
    "high_availability": true,
    "monitoring_required": true,
    "confidence": 0.95
  },
  "requires_human_review": true,
  "review_reason": "Production environment, GKE requires review"
}



## 8. Conclusion & Lessons Learned

### Conclusion

In this notebook, we implemented an LLM-powered GCP Architecture Requirements Processor using Groq, JSON, and Pydantic.

The solution converts natural-language cloud architecture requirements into structured, validated Python objects.

Our final workflow:

1. Receive natural-language GCP architecture requirements.
2. Extract structured information using an LLM.
3. Generate JSON output.
4. Validate the output against a Pydantic schema.
5. Detect and correct invalid responses using retry logic.
6. Process multiple architecture requirements.
7. Evaluate extraction accuracy against expected results.
8. Apply deterministic human-review policies.

All three architecture examples passed our evaluation (3/3).

### Lessons Learned

**1. JSON is not the same as validated data.**

An LLM may generate syntactically correct JSON while violating our application's data contract.

Pydantic helps enforce required fields, data types, allowed values, and constraints.

**2. Structured Outputs improve reliability.**

Compared with plain prompting, JSON Mode produced output that matched our expected structure in our tests.

However, JSON Mode alone does not guarantee schema compliance.

**3. Validation and error handling are essential.**

We tested two important failure scenarios:

- Invalid JSON syntax, such as trailing commas.
- Valid JSON containing invalid schema values.

Both must be handled before passing LLM responses to downstream systems.

**4. Retry mechanisms should use validation feedback.**

Instead of repeating the same request, we provided the LLM with the validation errors and original architecture requirements.

The model successfully corrected an invalid response during the first retry.

Retries should always be limited to control API usage, latency, and costs.

**5. Schema validation is not architecture validation.**

Pydantic validates the structure of architecture requirements.

It does not verify whether a GCP architecture is secure, highly available, cost-efficient, or production-ready.

Additional architecture policies and technical checks are required.

**6. LLM confidence is not a reliability guarantee.**

The confidence value returned by the model is self-reported.

A value of 0.95 or 1.0 does not prove that the extracted architecture is correct.

Independent evaluation and validation remain necessary.

**7. Human review should be policy-driven.**

We implemented deterministic review rules:

- Production architectures require human review.
- Architectures using GKE require human review.
- Low-confidence outputs require human review.

These rules provide a transparent decision mechanism independent of the LLM.

**8. Evaluation is different from validation.**

Validation checks whether an output follows the schema.

Evaluation checks whether the extracted information matches expected requirements.

Our three test cases passed, but a larger and more diverse evaluation dataset would be required to measure real-world reliability.

### Limitations

The current implementation is a learning prototype.

It does not yet include:

- GCP architecture best-practice validation.
- IAM, networking, and security policy checks.
- FinOps and resource cost estimation.
- Provider timeout and API failure handling.
- Production-grade observability and logging.
- Automated Terraform plan validation.
- Human approval workflows.

### Next Steps

Potential improvements include:

1. Extend the architecture schema with networking, security, and cost requirements.
2. Add deterministic architecture policy checks.
3. Integrate GCP resource and pricing information.
4. Introduce structured logging and evaluation metrics.
5. Build a controlled integration with Terraform planning.
6. Add an approval step before infrastructure deployment.

### Final Takeaway

**LLMs should extract and interpret requirements, while deterministic validation and policy mechanisms control how their outputs are used.**

This separation is a fundamental design principle for building reliable AI-assisted cloud architecture workflows.
